In [0]:
spark.sql(
    "DROP TABLE IF EXISTS workspace.gold.dim_event_type"
)

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.dim_event_type (
    event_type_key BIGINT,
    event_type STRING,
    event_type_description STRING
)
""")

In [0]:
from pyspark.sql import functions as F

event_type = (
    spark.table(
        "workspace.silver.usgs_earthquakes"
    )
    .select("event_type")
    .dropDuplicates()
    .withColumn(
        "event_type_description",
        F.when(
            F.col("event_type") == "earthquake",
            "Terremoto"
        )
        .when(
            F.col("event_type") == "quarry blast",
            "Explosión en cantera"
        )
        .when(
            F.col("event_type") == "explosion",
            "Explosión"
        )
        .otherwise(
            F.initcap(
                F.regexp_replace(
                    F.col("event_type"),
                    "_",
                    " "
                )
            )
        )
    )
    .withColumn(
        "event_type_key",
        F.xxhash64("event_type")
    )
)

event_type = event_type.select(
    "event_type_key",
    "event_type",
    "event_type_description"
)

In [0]:
event_type.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.gold.dim_event_type")